# Bane: End-to-End Execution-Aligned Text-to-SQL Pipeline

This notebook contains the complete end-to-end pipeline, **now with automated Email Alerting** for background runs.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets

### 0. Automated Email Alert System
Uses Kaggle Secrets to securely send you an email if the notebook crashes or succeeds.

In [ ]:
import smtplib
from email.mime.text import MIMEText
import traceback
from kaggle_secrets import UserSecretsClient

def send_email_alert(subject, body):
    try:
        user_secrets = UserSecretsClient()
        # You must configure these in the Kaggle 'Add-ons -> Secrets' menu
        sender_email = user_secrets.get_secret("EMAIL_ADDRESS")
        app_password = user_secrets.get_secret("EMAIL_PASSWORD")
        receiver_email = sender_email # Sending to yourself
        
        msg = MIMEText(body)
        msg['Subject'] = subject
        msg['From'] = sender_email
        msg['To'] = receiver_email
        
        # Assuming using Gmail SMTP
        server = smtplib.SMTP_SSL('smtp.gmail.com', 465)
        server.login(sender_email, app_password)
        server.sendmail(sender_email, receiver_email, msg.as_string())
        server.quit()
        print("\n[INFO] Alert email sent successfully!")
    except Exception as e:
        print(f"\n[WARNING] Failed to send email alert. Error: {e}")

### 1. Main Pipeline (Wrapped in Error Logging)

In [ ]:
import sqlite3
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel
from trl import DPOTrainer, DPOConfig
from transformers import TrainingArguments
import torch

def validate_sql_execution(predicted_sql: str, ground_truth_sql: str, table_schema: str) -> bool:
    try:
        conn = sqlite3.connect(':memory:')
        cursor = conn.cursor()
        for statement in table_schema.split(';'):
            if statement.strip():
                cursor.execute(statement)
        cursor.execute(predicted_sql)
        pred_res = cursor.fetchall()
        cursor.execute(ground_truth_sql)
        gt_res = cursor.fetchall()
        conn.close()
        return True 
    except:
        return False

def format_prompt(question, schema):
    return f"### Schema:\n{schema}\n\n### Question:\n{question}\n\n### SQL:\n"

# ---------------- MAIN EXECUTION BLOCK ---------------- #
try:
    print("Loading Base Model...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/llama-3-8b-Instruct",
        max_seq_length = 2048,
        dtype = None,
        load_in_4bit = True,
    )
    FastLanguageModel.for_inference(model)
    
    print("Generating execution-aligned dataset...")
    raw_dataset = load_dataset("b-mc2/sql-create-context", split="train[:500]")
    dpo_data = {"prompt": [], "chosen": [], "rejected": []}

    for i, row in enumerate(raw_dataset):
        prompt = format_prompt(row['question'], row['context'])
        ground_truth = row['answer']
        
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
        predicted_sql = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0].replace(prompt, "").strip()
        
        is_valid = validate_sql_execution(predicted_sql, ground_truth, row['context'])
        
        if not is_valid and predicted_sql != "":
            dpo_data["prompt"].append(prompt)
            dpo_data["chosen"].append(ground_truth)
            dpo_data["rejected"].append(predicted_sql)
            
    dpo_dataset = Dataset.from_dict(dpo_data)
    print(f"Final DPO Dataset Size: {len(dpo_dataset)} pairs")
    
    print("Starting DPO Training...")
    FastLanguageModel.for_training(model)
    model = FastLanguageModel.get_peft_model(
        model, r = 16, target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 16, lora_dropout = 0, bias = "none", use_gradient_checkpointing = "unsloth", random_state = 3407,
    )
    
    trainer = DPOTrainer(
        model = model,
        ref_model = None,
        tokenizer = tokenizer,
        beta = 0.1,
        train_dataset = dpo_dataset,
        args = DPOConfig(
            per_device_train_batch_size = 2, gradient_accumulation_steps = 4,
            warmup_ratio = 0.1, num_train_epochs = 1, learning_rate = 5e-6,
            fp16 = not torch.cuda.is_bf16_supported(), bf16 = torch.cuda.is_bf16_supported(),
            logging_steps = 1, optim = "adamw_8bit", output_dir = "outputs",
        ),
    )
    trainer.train()
    
    model.save_pretrained("bane_dpo_lora_adapters")
    tokenizer.save_pretrained("bane_dpo_lora_adapters")
    
    # IF EVERYTHING SUCCEEDS, SEND SUCCESS EMAIL
    send_email_alert("✅ Kaggle Bane Agent Complete!", f"The model finished training successfully.\nDataset size generated: {len(dpo_dataset)} pairs.")
    
except Exception as e:
    # IF ANYTHING FAILS, CATCH IT AND SEND THE ERROR LOG
    error_trace = traceback.format_exc()
    print(error_trace)
    send_email_alert("🚨 Kaggle Notebook FAILED", f"Your notebook crashed!\n\nError Log:\n{error_trace}")
    raise e # Re-raise the error so Kaggle registers the failure
